# Análisis de la Solución Secuencial

Este cuaderno reconstruye la solución secuencial del Laboratorio 3 (clasificador de letras del alfabeto ASL) y mide su tiempo de ejecución. Corresponde a los pasos 1 y 2 del proyecto. Todo el procesamiento se ejecuta en el CPU, y la GPU se oculta de forma deliberada para obtener una línea base de comparación.

El flujo se divide en dos fases. La **preparación de datos** lee, redimensiona y normaliza las imágenes. El **entrenamiento del modelo** ajusta una red neuronal convolucional (CNN, un tipo de red que aprende filtros sobre regiones de la imagen).

## Configuración del Entorno

La siguiente celda oculta la GPU mediante la variable `CUDA_VISIBLE_DEVICES`. Esta variable debe definirse antes de importar TensorFlow, por lo que se fija en la primera celda.

In [ ]:
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from src import config
from src.data.make_dataset import class_names, make_dataset
from src.features.build_features import preprocess_sequential, profile_stages
from src.models.train_model import build_cnn, device_name, evaluate, set_seed, train
from src.utils.monitor import ResourceMonitor
from src.utils.system_info import print_report
from src.utils.timing import Timer, save_times, summarize

print("Dispositivo de TensorFlow:", device_name())

### Hardware y Software Disponibles

El reporte describe el CPU, la memoria y las versiones de las librerías. Los hilos de TensorFlow se dejan en su valor por defecto (0 indica que TensorFlow usa todos los núcleos disponibles), tal como se ejecutaba el Laboratorio 3.

In [ ]:
print_report(include_nvidia_smi=False)
print()
print("Hilos intra-op de TensorFlow:", tf.config.threading.get_intra_op_parallelism_threads())
print("Hilos inter-op de TensorFlow:", tf.config.threading.get_inter_op_parallelism_threads())

## Preparación de los Datos

Se descarga el dataset ASL Alphabet de Kaggle y se toma una submuestra aleatoria de 500 imágenes por clase. Luego se divide de forma estratificada (cada subconjunto conserva la proporción de clases) en entrenamiento (70 %), validación (15 %) y prueba (15 %). El resultado se guarda en un manifiesto, un archivo CSV con la ruta, la clase y el subconjunto de cada imagen.

In [ ]:
manifest = make_dataset()
classes = class_names(manifest)

print("Imágenes en la submuestra:", len(manifest))
print("Número de clases:", len(classes))
manifest.groupby("split").size().rename("imágenes").to_frame()

In [ ]:
pd.crosstab(manifest["label"], manifest["split"]).describe().loc[["mean", "min", "max"]]

## Descomposición del Programa

Las siguientes celdas cuantifican las operaciones del programa. Primero se mide cuánto tarda cada etapa del preprocesamiento sobre todas las imágenes. Después se calcula cuántas veces se repite cada operación durante el entrenamiento.

In [ ]:
paths = manifest["path"].tolist()

with Timer() as t_first:
    X_all = preprocess_sequential(paths)
print(f"Primera ejecución del preprocesamiento (lectura de disco en frío): {t_first.elapsed:.2f} s")

stages = profile_stages(paths)
stages_df = pd.DataFrame({"segundos": stages})
stages_df["porcentaje"] = 100 * stages_df["segundos"] / stages_df["segundos"].sum()
stages_df["ms por imagen"] = 1000 * stages_df["segundos"] / len(paths)
stages_df.round(3)

In [ ]:
y_all = manifest["label_idx"].to_numpy()
masks = {s: (manifest["split"] == s).to_numpy() for s in ("train", "val", "test")}
X_train, y_train = X_all[masks["train"]], y_all[masks["train"]]
X_val, y_val = X_all[masks["val"]], y_all[masks["val"]]
X_test, y_test = X_all[masks["test"]], y_all[masks["test"]]

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)
print(f"Memoria de X_all: {X_all.nbytes / 1024**2:.1f} MB")

In [ ]:
set_seed()
model = build_cnn(n_classes=len(classes))
model.summary()

In [ ]:
def layer_macs(layer):
    """Multiplicaciones-acumulaciones por imagen de una capa Conv2D o Dense."""
    if isinstance(layer, tf.keras.layers.Conv2D):
        _, h, w, c_out = layer.output.shape
        k_h, k_w = layer.kernel_size
        return h * w * k_h * k_w * layer.input.shape[-1] * c_out
    if isinstance(layer, tf.keras.layers.Dense):
        return layer.input.shape[-1] * layer.units
    return 0


macs = pd.DataFrame(
    [(l.name, type(l).__name__, layer_macs(l)) for l in model.layers if layer_macs(l)],
    columns=["capa", "tipo", "MACs por imagen"],
)
macs["porcentaje"] = (100 * macs["MACs por imagen"] / macs["MACs por imagen"].sum()).round(2)
macs

In [ ]:
n_train = len(X_train)
batches_per_epoch = math.ceil(n_train / config.BATCH_SIZE)
total_steps = config.EPOCHS * batches_per_epoch
macs_per_image = int(macs["MACs por imagen"].sum())

repetitions = pd.DataFrame(
    [
        ("Lectura y decodificación JPEG", "Archivo de 200x200x3", len(paths)),
        ("Redimensionado a 64x64", "Imagen de 200x200x3", len(paths)),
        ("Normalización a [0, 1]", "Arreglo de 64x64x3", len(paths)),
        ("Paso de entrenamiento (lote)", f"Lote de {config.BATCH_SIZE} imágenes", total_steps),
        ("Propagación hacia adelante por imagen", "Imagen de 64x64x3", config.EPOCHS * n_train),
        ("Multiplicaciones-acumulaciones por época", "Pesos y activaciones", macs_per_image * n_train),
    ],
    columns=["Operación repetitiva", "Datos que procesa", "Cantidad de repeticiones"],
)
print("Lotes por época:", batches_per_epoch)
repetitions

## Medición del Tiempo de Ejecución Secuencial

Cada fase se ejecuta varias veces bajo las mismas condiciones y se reporta el promedio. El preprocesamiento se repite 5 veces. El entrenamiento se repite 3 veces con 10 épocas fijas y sin detención temprana (early stopping, técnica que detiene el entrenamiento cuando la validación deja de mejorar), para que todas las ejecuciones realicen el mismo trabajo. Durante cada fase se registra el uso de cada núcleo del CPU.

### Preparación de Datos en un Solo Núcleo

In [ ]:
records = [{"phase": "preprocess", "device": "CPU-1core", "rep": 0, "seconds": t_first.elapsed, "note": "first"}]

with ResourceMonitor(interval=0.5) as mon_pre:
    for rep in range(1, config.N_REPS_PREPROCESS + 1):
        with Timer() as t:
            preprocess_sequential(paths)
        records.append({"phase": "preprocess", "device": "CPU-1core", "rep": rep, "seconds": t.elapsed, "note": ""})
        print(f"Repetición {rep}/{config.N_REPS_PREPROCESS}: {t.elapsed:.2f} s")

cpu_pre = mon_pre.to_dataframe()
cpu_pre.to_csv(config.RESULTS_DIR / "01_cpu_usage_preprocess.csv", index=False)

### Entrenamiento del Modelo en CPU

In [ ]:
epoch_records = []
cpu_train = None

for rep in range(1, config.N_REPS_TRAIN + 1):
    set_seed()
    model = build_cnn(n_classes=len(classes))
    with ResourceMonitor(interval=0.5) as mon:
        result = train(model, X_train, y_train, X_val, y_val)
    test_loss, test_acc, t_inf = evaluate(model, X_test, y_test)

    records.append({"phase": "train", "device": "CPU", "rep": rep, "seconds": result.total_time, "note": ""})
    records.append({"phase": "inference", "device": "CPU", "rep": rep, "seconds": t_inf, "note": ""})
    for epoch, sec in enumerate(result.epoch_times, start=1):
        epoch_records.append({
            "device": "CPU", "rep": rep, "epoch": epoch, "seconds": sec,
            "train_acc": result.history["accuracy"][epoch - 1],
            "val_acc": result.history["val_accuracy"][epoch - 1],
        })
    if cpu_train is None:
        cpu_train = mon.to_dataframe()
    print(f"Repetición {rep}: entrenamiento {result.total_time:.1f} s, "
          f"accuracy de prueba {test_acc:.4f}, inferencia {t_inf:.2f} s")

cpu_train.to_csv(config.RESULTS_DIR / "01_cpu_usage_train.csv", index=False)

### Resumen de Tiempos

In [ ]:
times_df = save_times(records, config.RESULTS_DIR / "sequential_times.csv")
epochs_df = save_times(epoch_records, config.RESULTS_DIR / "sequential_epochs.csv")

summary = pd.DataFrame({
    phase: summarize(g.loc[g["note"] != "first", "seconds"].tolist())
    for phase, g in times_df.groupby("phase", sort=False)
}).T.round(3)
summary

In [ ]:
epochs_df.pivot(index="epoch", columns="rep", values="seconds").round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

summary["mean"].plot.bar(yerr=summary["std"], ax=axes[0], color="#4C72B0", capsize=4)
axes[0].set_title("Tiempo promedio por fase (CPU)")
axes[0].set_ylabel("Segundos")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

for rep, g in epochs_df.groupby("rep"):
    axes[1].plot(g["epoch"], g["seconds"], marker="o", label=f"Repetición {rep}")
axes[1].set_title("Duración de cada época (CPU)")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Segundos")
axes[1].legend()

fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "01_sequential_times.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in [
    (axes[0], cpu_pre, "Preprocesamiento en un núcleo"),
    (axes[1], cpu_train, "Entrenamiento en CPU (repetición 1)"),
]:
    cores = df.filter(like="cpu_").T
    im = ax.imshow(cores, aspect="auto", cmap="viridis", vmin=0, vmax=100,
                   extent=[df["t"].min(), df["t"].max(), len(cores) - 0.5, -0.5])
    ax.set_title(title)
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Núcleo lógico")
fig.colorbar(im, ax=axes, label="Uso del núcleo (%)")
fig.savefig(config.FIGURES_DIR / "01_cpu_usage.png", dpi=150, bbox_inches="tight")
plt.show()

print("Uso promedio del CPU en el preprocesamiento: "
      f"{cpu_pre.filter(like='cpu_').mean(axis=1).mean():.1f} %")
print("Uso promedio del CPU en el entrenamiento: "
      f"{cpu_train.filter(like='cpu_').mean(axis=1).mean():.1f} %")